# Dataset cleaning and split construction

This notebook:
1. Loads the merged paired dataset.
2. Normalizes column names.
3. Applies rule-based filtering for missing IDs, generation errors, blank/short code-mixed text, invalid labels, and duplicate source IDs.
4. Produces the cleaned master dataset.
5. Creates stratified train/test splits.
6. Exports aligned English, code-mix, and Tamil evaluation views.

In [ ]:
# Cell 1: setup + config
# Note: change input_csv and output_root before running.

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

CONFIG = {
    "seed": 42,
    "input_csv": "data/raw/codemix_full_merged.csv",
    "output_root": "data/processed/",
    "columns": {
        "id_candidates": ["sourceid", "source_id", "id", "datasetindex", "dataset_index"],
        "english_candidates": ["textenglish", "text_english", "english", "original"],
        "codemix_candidates": ["textcodemix", "text_codemix", "codemix", "50_grammarforce_Tamil"],
        "tamil_candidates":  ["texttamil", "text_tamil","tamil", "text_tamil"],
        "label_candidates": ["label", "hatelabel", "hate_label"],
        "error_candidates": ["error", "generation_error", "err"],
        "source_dataset_candidates": ["dataset", "source_dataset", "dataset_name"]
    },
    "filters": {
        "min_codemix_chars": 10,
        "drop_rows_with_error": True,
        "drop_blank_english": True,
        "drop_blank_codemix": True,
        "drop_short_codemix": True,
        "drop_invalid_label": True,
        "drop_duplicate_sourceid": True
    },
    "split": {
        "train_size": 0.55,
        "test_size": 0.45
    }
}

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(CONFIG["seed"])

ROOT = Path(CONFIG["output_root"])
DIRS = {
    "audits": ROOT / "audits",
    "splits": ROOT / "splits",
    "excluded": ROOT / "excluded",
    "logs": ROOT / "logs"
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Input CSV :", CONFIG["input_csv"])
print("Output root:", ROOT)

In [ ]:
# Cell 2: helper functions

def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def normalize_columns(df, config):
    df = df.copy()

    id_col = first_existing_column(df, config["columns"]["id_candidates"])
    eng_col = first_existing_column(df, config["columns"]["english_candidates"])
    cm_col = first_existing_column(df, config["columns"]["codemix_candidates"])
    ta_col = first_existing_column(df, config["columns"]["tamil_candidates"])
    label_col = first_existing_column(df, config["columns"]["label_candidates"])
    err_col = first_existing_column(df, config["columns"]["error_candidates"])
    src_dataset_col = first_existing_column(df, config["columns"]["source_dataset_candidates"])

    required = {
        "sourceid": id_col,
        "textenglish": eng_col,
        "textcodemix": cm_col,
        "label": label_col
    }
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns: {missing}\nAvailable columns: {list(df.columns)}")

    rename_map = {
        id_col: "sourceid",
        eng_col: "textenglish",
        cm_col: "textcodemix",
        ta_col: "texttamil",
        label_col: "label"
    }
    if err_col is not None:
        rename_map[err_col] = "error"
    if src_dataset_col is not None:
        rename_map[src_dataset_col] = "source_dataset"

    df = df.rename(columns=rename_map)

    if "error" not in df.columns:
        df["error"] = ""
    if "source_dataset" not in df.columns:
        df["source_dataset"] = "unknown"

    df["sourceid"] = df["sourceid"].astype(str).str.strip()
    df["textenglish"] = df["textenglish"].fillna("").astype(str).str.strip()
    df["textcodemix"] = df["textcodemix"].fillna("").astype(str).str.strip()
    df["error"] = df["error"].fillna("").astype(str).str.strip()
    df["source_dataset"] = df["source_dataset"].fillna("unknown").astype(str).str.strip()
    df["label"] = pd.to_numeric(df["label"], errors="coerce")

    return df

def is_blank(x):
    return not isinstance(x, str) or x.strip() == ""

def is_short_text(x, min_chars):
    if not isinstance(x, str):
        return True
    return len(x.strip()) < min_chars

def build_quality_flags(df, min_codemix_chars=10):
    out = df.copy()
    out["flag_missing_sourceid"] = out["sourceid"].eq("")
    out["flag_blank_english"] = out["textenglish"].apply(is_blank)
    out["flag_blank_codemix"] = out["textcodemix"].apply(is_blank)
    out["flag_short_codemix"] = out["textcodemix"].apply(lambda x: is_short_text(x, min_codemix_chars))
    out["flag_same_as_english"] = out["textenglish"].str.lower() == out["textcodemix"].str.lower()
    out["flag_blank_tamil"] = out["texttamil"].apply(is_blank)
    out["flag_same_tamil_as_english"] = out["textenglish"].str.lower() == out["texttamil"].str.lower()
    out["flag_short_tamil"] = out["texttamil"].apply(lambda x: is_short_text(x, min_codemix_chars))
    out["flag_error_nonempty"] = out["error"].str.strip().ne("")
    out["flag_duplicate_sourceid"] = out["sourceid"].duplicated(keep=False)
    out["flag_invalid_label"] = ~out["label"].isin([0, 1])
    return out

def summarize_dataset(df, name):
    return pd.DataFrame([{
        "dataset_name": name,
        "rows": len(df),
        "unique_sourceid": df["sourceid"].nunique(),
        "label_0_count": int((df["label"] == 0).sum()),
        "label_1_count": int((df["label"] == 1).sum()),
        "blank_english_rows": int(df["flag_blank_english"].sum()),
        "blank_codemix_rows": int(df["flag_blank_codemix"].sum()),
        "short_codemix_rows": int(df["flag_short_codemix"].sum()),
        "same_as_english_rows": int(df["flag_same_as_english"].sum()),
        "error_rows": int(df["flag_error_nonempty"].sum()),
        "duplicate_sourceid_rows": int(df["flag_duplicate_sourceid"].sum()),
        "invalid_label_rows": int(df["flag_invalid_label"].sum())
    }])

def make_view_df(df, text_col, view_name):
    out = df[["sourceid", "label", "textenglish", "textcodemix", "texttamil", "source_dataset"]].copy()
    out["text"] = out[text_col].astype(str).str.strip()
    out["view_name"] = view_name
    return out

def stratified_train_test_split(df, seed=42, train_size=0.55, test_size=0.45):
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["label"]
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

def overlap_count(df1, df2, key="sourceid"):
    return len(set(df1[key]).intersection(set(df2[key])))

In [ ]:
# Cell 3: load + normalize
# Note: after this cell we use only standardized column names.

input_path = Path(CONFIG["input_csv"])
if not input_path.exists():
    raise FileNotFoundError(f"Input CSV not found: {input_path}")

raw_df = pd.read_csv(input_path)
df = normalize_columns(raw_df, CONFIG)

print("Raw shape       :", raw_df.shape)
print("Normalized shape:", df.shape)
print("\nColumns:")
print(list(df.columns))

display(df.head(3))

In [ ]:
# Cell 4: audit + clean

flagged_df = build_quality_flags(df, min_codemix_chars=CONFIG["filters"]["min_codemix_chars"])
audit_before = summarize_dataset(flagged_df, "merged_raw_normalized")

exclude_mask = pd.Series(False, index=flagged_df.index)

if CONFIG["filters"]["drop_rows_with_error"]:
    exclude_mask |= flagged_df["flag_error_nonempty"]
if CONFIG["filters"]["drop_blank_english"]:
    exclude_mask |= flagged_df["flag_blank_english"]
if CONFIG["filters"]["drop_blank_codemix"]:
    exclude_mask |= flagged_df["flag_blank_codemix"]
if CONFIG["filters"]["drop_short_codemix"]:
    exclude_mask |= flagged_df["flag_short_codemix"]
if CONFIG["filters"]["drop_invalid_label"]:
    exclude_mask |= flagged_df["flag_invalid_label"]
if CONFIG["filters"]["drop_duplicate_sourceid"]:
    exclude_mask |= flagged_df["flag_duplicate_sourceid"]

exclude_mask |= flagged_df["flag_missing_sourceid"]

excluded_rows = flagged_df[exclude_mask].copy().reset_index(drop=True)
clean_df = flagged_df[~exclude_mask].copy().reset_index(drop=True)
clean_df["label"] = clean_df["label"].astype(int)

audit_after = summarize_dataset(clean_df, "merged_clean")
audit_all = pd.concat([audit_before, audit_after], ignore_index=True)

print("Excluded rows:", excluded_rows.shape)
print("Clean rows   :", clean_df.shape)
print("\nClean label distribution:")
print(clean_df["label"].value_counts(dropna=False))

display(audit_all)
display(clean_df.head(3))

In [ ]:
# Cell 5: split + save

train_master, test_master = stratified_train_test_split(
    clean_df,
    seed=CONFIG["seed"],
    train_size=CONFIG["split"]["train_size"],
    test_size=CONFIG["split"]["test_size"]
)

assert overlap_count(train_master, test_master) == 0

train_english = make_view_df(train_master, "textenglish", "train_english")
test_english = make_view_df(test_master, "textenglish", "test_english")
test_codemix = make_view_df(test_master, "textcodemix", "test_codemix")
test_tamil = make_view_df(test_master, "texttamil", "test_tamil")

audit_all.to_csv(DIRS["audits"] / "dataset_summary.csv", index=False)
clean_df.to_csv(DIRS["audits"] / "master_clean.csv", index=False)
excluded_rows.to_csv(DIRS["excluded"] / "excluded_rows.csv", index=False)

train_master.to_csv(DIRS["splits"] / "train_master.csv", index=False)
test_master.to_csv(DIRS["splits"] / "test_master.csv", index=False)
train_english.to_csv(DIRS["splits"] / "train_english.csv", index=False)
test_english.to_csv(DIRS["splits"] / "test_english.csv", index=False)
test_codemix.to_csv(DIRS["splits"] / "test_codemix.csv", index=False)
test_tamil.to_csv(DIRS["splits"] / "test_tamil.csv", index=False)

split_summary = pd.DataFrame([
    {"split": "train_master", "rows": len(train_master), "label_0": int((train_master["label"] == 0).sum()), "label_1": int((train_master["label"] == 1).sum())},
    {"split": "test_master", "rows": len(test_master), "label_0": int((test_master["label"] == 0).sum()), "label_1": int((test_master["label"] == 1).sum())},
    {"split": "train_english", "rows": len(train_english), "label_0": int((train_english["label"] == 0).sum()), "label_1": int((train_english["label"] == 1).sum())},
    {"split": "test_english", "rows": len(test_english), "label_0": int((test_english["label"] == 0).sum()), "label_1": int((test_english["label"] == 1).sum())},
    {"split": "test_codemix", "rows": len(test_codemix), "label_0": int((test_codemix["label"] == 0).sum()), "label_1": int((test_codemix["label"] == 1).sum())},
    {"split": "test_tamil", "rows": len(test_tamil), "label_0": int((test_tamil["label"] == 0).sum()), "label_1": int((test_tamil["label"] == 1).sum())}
])

split_summary.to_csv(DIRS["logs"] / "split_summary.csv", index=False)

run_log = pd.DataFrame([{
    "input_csv": str(input_path),
    "output_root": str(ROOT),
    "raw_rows": len(raw_df),
    "clean_rows": len(clean_df),
    "excluded_rows": len(excluded_rows),
    "train_rows": len(train_master),
    "test_rows": len(test_master),
    "seed": CONFIG["seed"]
}])

run_log.to_csv(DIRS["logs"] / "run_log.csv", index=False)

print("Saved files in:", ROOT)
print("\nSplit summary:")
display(split_summary)

In [ ]:
print("\n--- Head of train_english.csv ---")
display(pd.read_csv(DIRS["splits"] / "train_english.csv").head())

print("\n--- Head of test_english.csv ---")
display(pd.read_csv(DIRS["splits"] / "test_english.csv").head())

print("\n--- Head of test_codemix.csv ---")
display(pd.read_csv(DIRS["splits"] / "test_codemix.csv").head())

print("\n--- Head of train_master.csv ---")
display(pd.read_csv(DIRS["splits"] / "train_master.csv").head())

print("\n--- Head of test_master.csv ---")
display(pd.read_csv(DIRS["splits"] / "test_master.csv").head())

print("\n--- Head of test_tamil.csv ---")
display(pd.read_csv(DIRS["splits"] / "test_tamil.csv").head())